# Dense Product Detection — Training & Evaluation

**Priority:** Train YOLOv11n on ALL data (train+val+test combined) for mobile deployment.

**Models trained:**
- YOLOv8m (25.9M) — baseline, trained on 25% sample ✓
- YOLOv11l (25.3M) — SAHI-enhanced, trained on 25% sample ✓
- YOLOv11n (2.6M) — mobile model, trained on 25% sample ✓
- **YOLOv11n (2.6M) — mobile model, training on ALL data ← CURRENT PRIORITY**

**Data:** `s3://nutritionell-data-225989332790/raw/sku110k/SKU110K_fixed/`
**Output:** `s3://nutritionell-data-225989332790/models/sku110k/`

---
## 1. Setup

In [1]:
!pip install -q "ultralytics>=8.1,<8.5" "sahi>=0.11" "supervision>=0.18" "seaborn" "numpy<2.4"
import ultralytics; print(f"Ultralytics: {ultralytics.__version__}")
import sahi; print(f"SAHI: {sahi.__version__}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
sagemaker-serve 1.11.0 requires onnxruntime, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.3.5 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
gluonts 0.16.2 requires numpy<2.2,>=1.

WARNING ⚠️ torchvision==0.24 is incompatible with torch==2.8.
Run 'pip install torchvision==0.23' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation


Ultralytics: 8.4.83
SAHI: 0.12.1


In [2]:
import os, gc, json, time, shutil, random, pickle, io
from pathlib import Path
import boto3, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO
import supervision as sv
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:False"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"

S3_BUCKET = "nutritionell-data-225989332790"
S3_DATA_PREFIX = "raw/sku110k/SKU110K_fixed"
S3_OUTPUT_PREFIX = "models/sku110k"

HOME_DIR = Path("/home/sagemaker-user") if Path("/home/sagemaker-user").exists() else Path("/home/ec2-user/SageMaker")
PROJECT_ROOT = HOME_DIR / "sku110k_training"
DATA_DIR = PROJECT_ROOT / "SKU110K_fixed"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
for d in [PROJECT_ROOT, RUNS_DIR, RESULTS_DIR]: d.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = "0"
    GPU_NAME = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    GPU_MEM = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
else:
    DEVICE, GPU_NAME, GPU_MEM = "cpu", "CPU", 0
    print("⚠ No GPU.\n")

IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 20
s3 = boto3.client("s3")

# All model weight paths
YOLOV8_WEIGHTS = RUNS_DIR / "yolov8m" / "weights" / "best.pt"
YOLOV11_WEIGHTS = RUNS_DIR / "yolov11l" / "weights" / "best.pt"
YOLOV11X_WEIGHTS = RUNS_DIR / "yolov11x" / "weights" / "best.pt"
YOLOV11N_WEIGHTS = RUNS_DIR / "yolov11n" / "weights" / "best.pt"
YOLOV11N_ALL_DIR = RUNS_DIR / "yolov11n_all"
YOLOV11N_ALL_WEIGHTS = YOLOV11N_ALL_DIR / "weights" / "best.pt"
YOLOV11N_ALL_LAST = YOLOV11N_ALL_DIR / "weights" / "last.pt"

def is_fully_trained(run_dir, expected_epochs=EPOCHS):
    best = Path(run_dir) / "weights" / "best.pt"
    results = Path(run_dir) / "results.csv"
    if not best.exists() or not results.exists(): return False
    try: return (len(results.read_text().strip().split('\n')) - 1) >= 10
    except: return False

def clean_incomplete(run_dir):
    run_dir = Path(run_dir)
    if not run_dir.exists(): return
    rc = run_dir / "results.csv"
    if rc.exists(): print(f"  Deleting incomplete run ({len(rc.read_text().strip().split(chr(10)))-1} epochs)")
    else: print("  Deleting incomplete run")
    shutil.rmtree(run_dir)

print(f"Device: {DEVICE} — {GPU_NAME}" + (f" ({GPU_MEM:.0f} GB)" if GPU_MEM else ""))
print(f"Batch: {BATCH_SIZE}, Epochs: {EPOCHS}")
print(f"PyTorch: {torch.__version__}\n")

# Check existing models
for name, wp in [("yolov8m",YOLOV8_WEIGHTS),("yolov11l",YOLOV11_WEIGHTS),("yolov11n",YOLOV11N_WEIGHTS),("yolov11n_all",YOLOV11N_ALL_WEIGHTS)]:
    if wp.exists():
        rd = wp.parent.parent
        rc = rd / "results.csv"
        n = len(rc.read_text().strip().split('\n'))-1 if rc.exists() else '?'
        print(f"  ✓ {name}: best.pt ({n} epochs)")
    else:
        print(f"  ⊘ {name}: not trained")

Device: 0 — NVIDIA A10G (24 GB)
Batch: 8, Epochs: 20
PyTorch: 2.8.0

  ✓ yolov8m: best.pt (24 epochs)
  ✓ yolov11l: best.pt (20 epochs)
  ✓ yolov11n: best.pt (20 epochs)
  ✓ yolov11n_all: best.pt (47 epochs)


---
## 2. Sync Data from S3

In [3]:
def s3_sync(bucket, prefix, local_dir):
    local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)
    paginator = s3.get_paginator("list_objects_v2")
    all_keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []): all_keys.append(obj["Key"])
    print(f"Found {len(all_keys):,} files")
    downloaded, skipped = 0, 0
    for key in tqdm(all_keys, desc="Syncing"):
        relative = key[len(prefix):].lstrip("/")
        if not relative: continue
        local_path = local_dir / relative
        if local_path.exists(): skipped += 1; continue
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(local_path))
        downloaded += 1
    print(f"Done: {downloaded:,} new, {skipped:,} existed")

if (DATA_DIR/"images"/"train").exists() and len(list((DATA_DIR/"images"/"train").glob("*.jpg"))) > 100:
    print(f"Dataset exists at {DATA_DIR}")
else:
    s3_sync(S3_BUCKET, S3_DATA_PREFIX, DATA_DIR)

Dataset exists at /home/sagemaker-user/sku110k_training/SKU110K_fixed


In [4]:
# Verify + resync missing labels
total_imgs, total_boxes = 0, 0
for split in ["train","val","test"]:
    img_dir, lbl_dir = DATA_DIR/"images"/split, DATA_DIR/"labels"/split
    imgs = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0
    lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    boxes = sum(len(f.read_text().strip().split('\n')) for f in lbl_dir.glob("*.txt") if f.read_text().strip()) if lbl_dir.exists() else 0
    s3c = sum(len(p.get("Contents",[])) for p in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=f"{S3_DATA_PREFIX}/labels/{split}/"))
    total_imgs += imgs; total_boxes += boxes
    ok = "✓" if lbls >= s3c and lbls > 0 else "✗"
    print(f"  {ok} {split:6s} {imgs:,} imgs  {lbls:,} lbls  {boxes:,} boxes" + (f"  (S3:{s3c})" if lbls < s3c else ""))
    if lbls < s3c: s3_sync(S3_BUCKET, f"{S3_DATA_PREFIX}/labels/{split}", DATA_DIR/"labels"/split)
print(f"  Total: {total_imgs:,} images, {total_boxes:,} boxes")

  ✓ train  8,185 imgs  8,185 lbls  1,155,010 boxes


  ✓ val    584 imgs  584 lbls  90,456 boxes


  ✓ test   2,920 imgs  2,920 lbls  429,411 boxes
  Total: 11,689 images, 1,674,877 boxes


---
## 3. Train YOLOv11n on ALL Data (Priority)

Combines train+val+test into one training set (~11,700 images). Crash-safe: re-run this cell after kernel restart and it resumes from the last checkpoint.

In [5]:
# Create combined dataset (all splits merged)
combined_img = DATA_DIR / "images" / "all_combined"
combined_lbl = DATA_DIR / "labels" / "all_combined"
combined_img.mkdir(parents=True, exist_ok=True)
combined_lbl.mkdir(parents=True, exist_ok=True)

if len(list(combined_img.glob("*.jpg"))) > 1000:
    print(f"Combined set exists: {len(list(combined_img.glob('*.jpg')))} images")
else:
    total = 0
    for split in ["train","val","test"]:
        for img in (DATA_DIR/"images"/split).glob("*.jpg"):
            dst_img = combined_img / f"{split}_{img.name}"
            dst_lbl = combined_lbl / f"{split}_{img.stem}.txt"
            src_lbl = DATA_DIR/"labels"/split/(img.stem+".txt")
            if not dst_img.exists(): os.symlink(img.resolve(), dst_img)
            if src_lbl.exists() and not dst_lbl.exists(): os.symlink(src_lbl.resolve(), dst_lbl)
            total += 1
    print(f"Combined: {total} images symlinked")

yaml_all = DATA_DIR / "dataset_all.yaml"
yaml_all.write_text(f"path: {DATA_DIR}\ntrain: images/all_combined\nval: images/val\ntest: images/test\n\nnames:\n  0: product\n")
for c in DATA_DIR.rglob("*.cache"): c.unlink()
print(f"Config: {yaml_all}")

Combined set exists: 11689 images
Config: /home/sagemaker-user/sku110k_training/SKU110K_fixed/dataset_all.yaml


In [6]:
# # Delete the incomplete run and retrain
# shutil.rmtree(YOLOV11N_ALL_DIR)
# for c in DATA_DIR.rglob("*.cache"): c.unlink()
# print("Deleted incomplete run + caches")

In [7]:
# Train YOLOv11n on ALL data — 10 epochs at a time with memory clearing
# Re-run this cell after any crash — it picks up where it left off.

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

FULL_EPOCHS = 50
CHUNK_SIZE = 2
BATCH = 8

yaml_all_str = str(yaml_all)

def get_completed_epochs():
    rc = YOLOV11N_ALL_DIR / "results.csv"
    if not rc.exists(): return 0
    content = rc.read_text().strip()
    if not content: return 0
    return len(content.split('\n')) - 1

completed = get_completed_epochs()
print(f"Status: {completed}/{FULL_EPOCHS} epochs completed")

if completed >= FULL_EPOCHS:
    print(f"✓ Fully trained ({completed} epochs)")
else:
    while completed < FULL_EPOCHS:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        import ctypes
        try: ctypes.CDLL("libc.so.6").malloc_trim(0)
        except: pass
        
        target = min(completed + CHUNK_SIZE, FULL_EPOCHS)
        
        print(f"\n{'='*60}")
        print(f"  Chunk: epochs {completed+1}–{target} of {FULL_EPOCHS}")
        print(f"{'='*60}\n")
        
        try:
            if YOLOV11N_ALL_LAST.exists():
                model = YOLO(str(YOLOV11N_ALL_LAST))
                model.train(
                    data=yaml_all_str, epochs=FULL_EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
                    project=str(RUNS_DIR), name="yolov11n_all",
                    mosaic=0.0, mixup=0.0, copy_paste=0.0, scale=0.5, fliplr=0.5, flipud=0.0,
                    patience=FULL_EPOCHS, save=True, save_period=1,
                    device=DEVICE, workers=0, verbose=True,
                    resume=True, exist_ok=True,
                )
            else:
                model = YOLO("yolo11n.pt")
                model.train(
                    data=yaml_all_str, epochs=FULL_EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
                    project=str(RUNS_DIR), name="yolov11n_all",
                    mosaic=0.0, mixup=0.0, copy_paste=0.0, scale=0.5, fliplr=0.5, flipud=0.0,
                    patience=FULL_EPOCHS, save=True, save_period=1,
                    device=DEVICE, workers=0, verbose=True,
                    exist_ok=True,
                )
            
            # If we get here, training ran to FULL_EPOCHS (won't normally happen in chunks)
            del model; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            completed = get_completed_epochs()
            break
            
        except KeyboardInterrupt:
            print("\nManually stopped.")
            try: del model
            except: pass
            break
            
        except Exception as e:
            try: del model
            except: pass
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            try: ctypes.CDLL("libc.so.6").malloc_trim(0)
            except: pass
            
            completed = get_completed_epochs()
            print(f"\n⚠ Crashed after epoch {completed}")
            
            # Delete any duplicate run directories (yolov11n_all-2, -3, etc.)
            for dup in RUNS_DIR.glob("yolov11n_all-*"):
                shutil.rmtree(dup)
                print(f"  Cleaned up duplicate: {dup.name}")
            
            if completed >= FULL_EPOCHS:
                break
            
            print(f"  {completed}/{FULL_EPOCHS} saved. Retrying in 10s...\n")
            time.sleep(10)
            continue
    
    completed = get_completed_epochs()
    print(f"\n{'='*60}")
    print(f"  ✓ {completed}/{FULL_EPOCHS} epochs completed")
    print(f"{'='*60}")

Status: 47/50 epochs completed



  Chunk: epochs 48–49 of 50



Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.8.0 CUDA:0 (NVIDIA A10G, 22590MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sagemaker-user/sku110k_training/SKU110K_fixed/dataset_all.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/sagemaker-user/sku110k_training/runs/yolov11n_all/weights/last.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov11n_all, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_


                   from  n    params  module                                       arguments                     


...

                   all        584      90456      0.907      0.833      0.885      0.544


Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.8ms postprocess per image


Results saved to /home/sagemaker-user/sku110k_training/runs/yolov11n_all


MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'



  ✓ 50/50 epochs completed


In [8]:
# Save final model + upload to S3
if YOLOV11N_ALL_WEIGHTS.exists():
    pkl_path = YOLOV11N_ALL_DIR / "yolov11n_all_final.pkl"
    model = YOLO(str(YOLOV11N_ALL_WEIGHTS))
    with open(pkl_path, "wb") as f: pickle.dump(model, f)
    print(f"✓ Pickle: {pkl_path} ({pkl_path.stat().st_size/1e6:.1f} MB)")
    print(f"✓ PyTorch: {YOLOV11N_ALL_WEIGHTS} ({YOLOV11N_ALL_WEIGHTS.stat().st_size/1e6:.1f} MB)")
    
    for artifact in YOLOV11N_ALL_DIR.rglob("*"):
        if artifact.is_dir(): continue
        if artifact.suffix in (".pt",".pkl",".csv",".yaml",".png"):
            rel = artifact.relative_to(YOLOV11N_ALL_DIR)
            s3.upload_file(str(artifact), S3_BUCKET, f"{S3_OUTPUT_PREFIX}/yolov11n_all/{rel}")
    print(f"\n✓ Uploaded to s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/yolov11n_all/")
    del model; gc.collect()
else:
    print("No best.pt — training may not have completed. Re-run the training cell.")

✓ Pickle: /home/sagemaker-user/sku110k_training/runs/yolov11n_all/yolov11n_all_final.pkl (10.7 MB)
✓ PyTorch: /home/sagemaker-user/sku110k_training/runs/yolov11n_all/weights/best.pt (5.4 MB)



✓ Uploaded to s3://nutritionell-data-225989332790/models/sku110k/yolov11n_all/


## Real-World Test: YOLOv11n on Nutritionell Images

Testing the trained model on real shelf images from `s3://nutritionell-images-225989332790/`. These are images the model has never seen — not from SKU-110K.

In [12]:
!pip install -q pillow-heif

from pillow_heif import register_heif_opener
register_heif_opener()  # Now PIL can open .heic files

TEST_BUCKET = "nutritionell-images-225989332790"
TEST_DIR = PROJECT_ROOT / "nutritionell_test_images"
TEST_DIR.mkdir(parents=True, exist_ok=True)

paginator = s3.get_paginator("list_objects_v2")
all_keys = []
for page in paginator.paginate(Bucket=TEST_BUCKET):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.lower().endswith((".jpg",".jpeg",".png",".webp",".heic",".heif")):
            all_keys.append(key)

print(f"Found {len(all_keys)} images in s3://{TEST_BUCKET}/\n")
for k in all_keys[:20]:
    print(f"  {k}")

downloaded = 0
for key in tqdm(all_keys, desc="Downloading"):
    local_path = TEST_DIR / key.replace("/", "_")
    if local_path.exists(): continue
    try:
        s3.download_file(TEST_BUCKET, key, str(local_path))
        downloaded += 1
    except Exception as e:
        print(f"  Skip {key}: {e}")

test_images = sorted([f for f in TEST_DIR.glob("*") if f.suffix.lower() in (".jpg",".jpeg",".png",".webp",".heic",".heif")])
print(f"\n{len(test_images)} images ready ({downloaded} new downloads)")

Found 47 images in s3://nutritionell-images-225989332790/

  IMG_6056.HEIC
  IMG_6057.HEIC
  IMG_6058.HEIC
  IMG_6059.HEIC
  IMG_6060.HEIC
  IMG_6061.HEIC
  IMG_6228 (1).HEIC
  IMG_6228.HEIC
  IMG_6229.HEIC
  IMG_6230.HEIC
  IMG_6231.HEIC
  IMG_6232.HEIC
  IMG_6233.HEIC
  IMG_6234.HEIC
  IMG_6235.HEIC
  IMG_6236.HEIC
  IMG_6237.HEIC
  IMG_6238.HEIC
  IMG_6239.HEIC
  IMG_6240.HEIC


Downloading:   0%|          | 0/47 [00:00<?, ?it/s]

Downloading:   2%|▏         | 1/47 [00:00<00:08,  5.34it/s]

Downloading:   4%|▍         | 2/47 [00:00<00:06,  6.49it/s]

Downloading:   6%|▋         | 3/47 [00:00<00:07,  5.69it/s]

...

Downloading:  96%|█████████▌| 45/47 [00:09<00:00,  4.05it/s]

Downloading:  98%|█████████▊| 46/47 [00:09<00:00,  4.72it/s]

Downloading: 100%|██████████| 47/47 [00:09<00:00,  4.24it/s]

Downloading: 100%|██████████| 47/47 [00:09<00:00,  4.87it/s]


47 images ready (47 new downloads)


In [13]:
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

pkl_path = YOLOV11N_ALL_DIR / "yolov11n_all_final.pkl"
if pkl_path.exists():
    with open(pkl_path, "rb") as f:
        model = pickle.load(f)
    print(f"✓ Loaded from pickle: {pkl_path.name}")
else:
    print("No pickle found — using best.pt instead")
    model = YOLO(str(YOLOV11N_ALL_WEIGHTS))

nutri_results = RESULTS_DIR / "nutritionell_test"
nutri_results.mkdir(parents=True, exist_ok=True)

def draw_box(img, x1, y1, x2, y2, color=(255, 0, 255), t=5):
    cv2.rectangle(img, (x1,y1), (x2,y2), (0,0,0), t+2)
    cv2.rectangle(img, (x1,y1), (x2,y2), color, t)

def load_image(path):
    """Load any image format (including HEIC) and return RGB numpy array."""
    pil_img = Image.open(path).convert("RGB")
    return np.array(pil_img)

# Convert HEIC to JPG for YOLO (YOLO can't read HEIC directly)
converted_dir = TEST_DIR / "converted"
converted_dir.mkdir(exist_ok=True)

def ensure_jpg(img_path):
    """Convert HEIC to JPG if needed, return path YOLO can read."""
    if img_path.suffix.lower() in (".heic", ".heif"):
        jpg_path = converted_dir / (img_path.stem + ".jpg")
        if not jpg_path.exists():
            pil_img = Image.open(img_path).convert("RGB")
            pil_img.save(jpg_path, "JPEG", quality=95)
        return jpg_path
    return img_path

summary_rows = []

for img_path in test_images:
    try:
        # Convert HEIC to JPG if needed
        yolo_path = ensure_jpg(img_path)
        img_rgb = load_image(img_path)
        h, w = img_rgb.shape[:2]
        name = img_path.stem

        disp = img_rgb.copy()
        res = model.predict(str(yolo_path), conf=0.25, iou=0.4, verbose=False, max_det=900)
        dets = sv.Detections.from_ultralytics(res[0])
        count = len(dets)
        for xyxy in dets.xyxy:
            draw_box(disp, *map(int, xyxy))

        fig, ax = plt.subplots(figsize=(14, 10))
        ax.imshow(disp); ax.axis("off")
        ax.set_title(f"{name} — YOLOv11n (all data) — {count} products", fontsize=16, fontweight="bold")
        plt.tight_layout()
        plt.savefig(nutri_results / f"{name}_{count}_products.png", dpi=150, bbox_inches="tight")
        plt.show()

        summary_rows.append({"image": name, "format": img_path.suffix, "size": f"{w}x{h}", "products_detected": count})
        print(f"  {name}: {count} products detected")
    except Exception as e:
        print(f"  ✗ {img_path.name}: {e}")

del model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

✓ Loaded from pickle: yolov11n_all_final.pkl


<Figure size 1400x1000 with 1 Axes>

  IMG_6056: 15 products detected


<Figure size 1400x1000 with 1 Axes>

  IMG_6057: 32 products detected


...

  IMG_6265: 101 products detected


<Figure size 1400x1000 with 1 Axes>

  IMG_6266: 112 products detected


<Figure size 1400x1000 with 1 Axes>

  IMG_6267: 103 products detected


In [14]:
summary_df = pd.DataFrame(summary_rows)
print("\nNutritionell Test Results:")
print("═" * 60)
print(summary_df.to_string(index=False))
print("═" * 60)
print(f"\nAverage: {summary_df['products_detected'].mean():.0f} products/image")
print(f"Min: {summary_df['products_detected'].min()}  Max: {summary_df['products_detected'].max()}")

summary_df.to_csv(nutri_results / "nutritionell_test_results.csv", index=False)

print("\nUploading to S3...")
for f in sorted(nutri_results.glob("*")):
    if f.is_dir(): continue
    s3_key = f"{S3_OUTPUT_PREFIX}/results/nutritionell_test/{f.name}"
    s3.upload_file(str(f), S3_BUCKET, s3_key)
    print(f"  {f.name}")
print(f"\nDownload: https://console.aws.amazon.com/s3/buckets/{S3_BUCKET}?prefix={S3_OUTPUT_PREFIX}/results/nutritionell_test/")


Nutritionell Test Results:
════════════════════════════════════════════════════════════
       image format      size  products_detected
    IMG_6056  .HEIC 4032x2268                 15
    IMG_6057  .HEIC 4032x2268                 32
    IMG_6058  .HEIC 4032x2268                 99
    IMG_6059  .HEIC 4032x2268                 29
    IMG_6060  .HEIC 4032x2268                 19
    IMG_6061  .HEIC 4032x2268                 24
IMG_6228 (1)  .HEIC 4032x2268                 51
    IMG_6228  .HEIC 4032x2268                 51
    IMG_6229  .HEIC 4032x2268                 78
    IMG_6230  .HEIC 4032x2268                 71
    IMG_6231  .HEIC 4032x2268                  1
    IMG_6232  .HEIC 2268x4032                  3
    IMG_6233  .HEIC 4032x2268                  4
    IMG_6234  .HEIC 4032x2268                  8
    IMG_6235  .HEIC 4032x2268                 11
    IMG_6236  .HEIC 4032x2268                 23
    IMG_6237  .HEIC 4032x2268                 14
    IMG_6238  .HEIC 4032x2268

  IMG_6057_32_products.png
  IMG_6058_99_products.png
  IMG_6059_29_products.png


  IMG_6060_19_products.png
  IMG_6061_24_products.png


  IMG_6228 (1)_51_products.png
  IMG_6228_51_products.png
  IMG_6229_78_products.png


  IMG_6230_71_products.png
  IMG_6231_1_products.png
  IMG_6232_3_products.png


...

  IMG_6258_193_products.png
  IMG_6259_191_products.png


  IMG_6260_232_products.png
  IMG_6261_190_products.png


  IMG_6262_184_products.png
  IMG_6263_81_products.png
  IMG_6264_110_products.png


  IMG_6265_101_products.png
  IMG_6266_112_products.png
  IMG_6267_103_products.png


  nutritionell_test_results.csv

Download: https://console.aws.amazon.com/s3/buckets/nutritionell-data-225989332790?prefix=models/sku110k/results/nutritionell_test/


---
## 4. Train Other Models (25% sample)

These use a 25% random sample of training images. Skip if already trained.

In [ ]:
# Create 25% sampled training set
SAMPLE_RATIO = 0.25
sampled_img = DATA_DIR / "images" / "train_sampled"
sampled_lbl = DATA_DIR / "labels" / "train_sampled"

if sampled_img.exists() and len(list(sampled_img.glob("*.jpg"))) > 100:
    print(f"Sample exists: {len(list(sampled_img.glob('*.jpg')))} images")
else:
    sampled_img.mkdir(parents=True, exist_ok=True)
    sampled_lbl.mkdir(parents=True, exist_ok=True)
    all_train = sorted((DATA_DIR/"images"/"train").glob("*.jpg"))
    random.seed(42)
    sampled = random.sample(all_train, int(len(all_train)*SAMPLE_RATIO))
    for img in tqdm(sampled, desc="Sampling"):
        dst_img = sampled_img / img.name
        dst_lbl = sampled_lbl / (img.stem+".txt")
        src_lbl = DATA_DIR/"labels"/"train"/(img.stem+".txt")
        if not dst_img.exists(): os.symlink(img.resolve(), dst_img)
        if src_lbl.exists() and not dst_lbl.exists(): os.symlink(src_lbl.resolve(), dst_lbl)
    print(f"Sampled: {len(list(sampled_img.glob('*.jpg')))} images")

yaml_path = DATA_DIR / "dataset.yaml"
yaml_path.write_text(f"path: {DATA_DIR}\ntrain: images/train_sampled\nval: images/val\ntest: images/test\n\nnames:\n  0: product\n")
for c in DATA_DIR.rglob("*.cache"): c.unlink()
print(f"Config: {yaml_path}")

### 4a. YOLOv8m (Baseline, 25.9M params)

In [ ]:
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
for c in DATA_DIR.rglob("*.cache"): c.unlink()

if is_fully_trained(RUNS_DIR/"yolov8m"):
    n = len((RUNS_DIR/"yolov8m"/"results.csv").read_text().strip().split('\n'))-1
    print(f"✓ YOLOv8m trained ({n} epochs)")
else:
    clean_incomplete(RUNS_DIR/"yolov8m")
    model = YOLO("yolov8m.pt")
    model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
        project=str(RUNS_DIR), name="yolov8m",
        mosaic=0.0, mixup=0.0, copy_paste=0.0, scale=0.5, fliplr=0.5, flipud=0.0,
        patience=10, save=True, save_period=5, device=DEVICE, workers=0, verbose=True)
    del model; gc.collect(); torch.cuda.empty_cache()

### 4b. YOLOv11l + SAHI (25.3M params)

In [ ]:
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

if is_fully_trained(RUNS_DIR/"yolov11l"):
    n = len((RUNS_DIR/"yolov11l"/"results.csv").read_text().strip().split('\n'))-1
    print(f"✓ YOLOv11l trained ({n} epochs)")
else:
    clean_incomplete(RUNS_DIR/"yolov11l")
    model = YOLO("yolo11l.pt")
    model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
        project=str(RUNS_DIR), name="yolov11l",
        mosaic=0.0, mixup=0.0, copy_paste=0.0, scale=0.5, fliplr=0.5, flipud=0.0,
        patience=10, save=True, save_period=5, device=DEVICE, workers=0, verbose=True)
    del model; gc.collect(); torch.cuda.empty_cache()

### 4c. YOLOv11n (Mobile, 2.6M params, 25% sample)

In [ ]:
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

if is_fully_trained(RUNS_DIR/"yolov11n"):
    n = len((RUNS_DIR/"yolov11n"/"results.csv").read_text().strip().split('\n'))-1
    print(f"✓ YOLOv11n trained ({n} epochs)")
else:
    clean_incomplete(RUNS_DIR/"yolov11n")
    model = YOLO("yolo11n.pt")
    model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
        project=str(RUNS_DIR), name="yolov11n",
        mosaic=0.0, mixup=0.0, copy_paste=0.0, scale=0.5, fliplr=0.5, flipud=0.0,
        patience=10, save=True, save_period=5, device=DEVICE, workers=0, verbose=True)
    del model; gc.collect(); torch.cuda.empty_cache()

---
## 5. Upload All Models to S3

In [ ]:
def upload_artifacts(run_name, local_dir):
    local_dir = Path(local_dir); uploaded = []
    for f in local_dir.rglob("*"):
        if f.is_dir(): continue
        if f.suffix in (".pt",".pkl",".csv",".yaml",".json",".png",".onnx",".tflite"):
            rel = f.relative_to(local_dir)
            s3.upload_file(str(f), S3_BUCKET, f"{S3_OUTPUT_PREFIX}/{run_name}/{rel}")
            uploaded.append((str(rel), f.stat().st_size/1e6))
    return uploaded

print("Uploading to S3...\n")
for name, w, rd in [("YOLOv8m",YOLOV8_WEIGHTS,RUNS_DIR/"yolov8m"),("YOLOv11l",YOLOV11_WEIGHTS,RUNS_DIR/"yolov11l"),("YOLOv11n",YOLOV11N_WEIGHTS,RUNS_DIR/"yolov11n"),("YOLOv11n_all",YOLOV11N_ALL_WEIGHTS,YOLOV11N_ALL_DIR)]:
    if w.exists():
        arts = upload_artifacts(rd.name, rd)
        print(f"  {name:15s} {len(arts)} files")
        for fn,mb in arts:
            if mb > 1: print(f"    {fn} ({mb:.0f} MB)")
print(f"\nSaved to s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/")

---
## 6. Validation Metrics

In [ ]:
gc.collect(); torch.cuda.empty_cache()
val_results = []
test_imgs = sorted((DATA_DIR/"images"/"test").glob("*.jpg"), key=lambda p: int(p.stem.split("_")[1]))[:20]

for model_name, wp in [("YOLOv8m",YOLOV8_WEIGHTS),("YOLOv11l (std)",YOLOV11_WEIGHTS),("YOLOv11n (sample)",YOLOV11N_WEIGHTS),("YOLOv11n (all)",YOLOV11N_ALL_WEIGHTS)]:
    if not wp.exists(): print(f"⊘ {model_name}"); continue
    gc.collect(); torch.cuda.empty_cache()
    print(f"Validating {model_name}...", end=" ")
    m = YOLO(str(wp))
    metrics = m.val(data=str(yaml_path), split="test", verbose=False, batch=4, max_det=900)
    t0 = time.time()
    for ip in test_imgs: m.predict(str(ip), conf=0.25, iou=0.4, verbose=False, max_det=900)
    ms = ((time.time()-t0)/len(test_imgs))*1000
    p, r = float(metrics.box.mp), float(metrics.box.mr)
    f1 = 2*p*r/(p+r) if (p+r)>0 else 0
    val_results.append({"Model":model_name,"mAP@50":round(float(metrics.box.map50),4),"mAP@50:95":round(float(metrics.box.map),4),
        "Precision":round(p,4),"Recall":round(r,4),"F1":round(f1,4),"Inf (ms)":round(ms,1),
        "Params (M)":round(sum(pa.numel() for pa in m.model.parameters())/1e6,1)})
    print(f"mAP@50={val_results[-1]['mAP@50']:.4f}  F1={f1:.3f}  {ms:.1f}ms")
    del m; gc.collect(); torch.cuda.empty_cache()

comparison_df = pd.DataFrame(val_results)
print("\n" + comparison_df.to_string(index=False))

---
## 7. Predicted vs Ground Truth Scatter Plots

In [ ]:
for split in ["val","test"]:
    img_dir = DATA_DIR/"images"/split
    lbl_dir = DATA_DIR/"labels"/split
    images = sorted(img_dir.glob("*.jpg"), key=lambda p: int(p.stem.split("_")[1]))[:20]
    scatter_data = []
    print(f"\n{'='*50}\n  {split.upper()} (first {len(images)} images)\n{'='*50}")
    for img_path in tqdm(images, desc=split):
        lbl = lbl_dir/(img_path.stem+".txt")
        gt = len(lbl.read_text().strip().split('\n')) if lbl.exists() and lbl.read_text().strip() else 0
        row = {"image":img_path.name, "ground_truth":gt}
        for mname, wp in [("YOLOv8m",YOLOV8_WEIGHTS),("YOLOv11l+SAHI",YOLOV11_WEIGHTS),("YOLOv11n",YOLOV11N_WEIGHTS),("YOLOv11n_all",YOLOV11N_ALL_WEIGHTS)]:
            if not wp.exists(): continue
            gc.collect(); torch.cuda.empty_cache()
            if "SAHI" in mname:
                sm = AutoDetectionModel.from_pretrained(model_type="ultralytics",model_path=str(wp),confidence_threshold=0.25,device=DEVICE if DEVICE!="0" else "cuda:0")
                res = get_sliced_prediction(str(img_path),sm,slice_height=640,slice_width=640,overlap_height_ratio=0.2,overlap_width_ratio=0.2,perform_standard_pred=True,postprocess_match_threshold=0.5,verbose=0)
                row[mname] = len(res.object_prediction_list)
                del sm
            else:
                m = YOLO(str(wp))
                res = m.predict(str(img_path),conf=0.25,iou=0.4,verbose=False,max_det=900)
                row[mname] = len(sv.Detections.from_ultralytics(res[0]))
                del m
            gc.collect(); torch.cuda.empty_cache()
        scatter_data.append(row)
    sdf = pd.DataFrame(scatter_data)
    fig, ax = plt.subplots(figsize=(10,8))
    colors = {"YOLOv8m":"#7c6aff","YOLOv11l+SAHI":"#38bdf8","YOLOv11n":"#fb923c","YOLOv11n_all":"#ff5c7a"}
    mcols = [c for c in sdf.columns if c not in ("image","ground_truth")]
    for col in mcols:
        ax.scatter(sdf["ground_truth"],sdf[col],s=80,alpha=0.7,label=col,color=colors.get(col,"gray"),edgecolors="white",linewidth=0.5,zorder=5)
    mx = max(sdf["ground_truth"].max(), max(sdf[c].max() for c in mcols))
    ax.plot([0,mx],[0,mx],'b--',alpha=0.3,linewidth=1.5,label="Perfect")
    ax.set_xlabel("Ground Truth Count",fontsize=16); ax.set_ylabel("Predicted Count",fontsize=16)
    ax.set_title(f"Predicted vs Ground Truth — First 20 {split.upper()}",fontsize=18,fontweight="bold")
    ax.legend(fontsize=14,loc="upper left"); ax.grid(True,alpha=0.15)
    plt.tight_layout(); plt.savefig(RESULTS_DIR/f"predicted_vs_gt_{split}.png",dpi=150,bbox_inches="tight"); plt.show()
    for col in mcols:
        diff = (sdf[col]-sdf["ground_truth"]).mean()
        print(f"    {col:20s}  avg diff: {diff:+.1f}")

---
## 8. Visual Comparison (Individual Images, PPT-ready)

In [ ]:
gc.collect(); torch.cuda.empty_cache()
val_img_dir = DATA_DIR/"images"/"val"
val_lbl_dir = DATA_DIR/"labels"/"val"
val_images = sorted(val_img_dir.glob("*.jpg"), key=lambda p: int(p.stem.split("_")[1]))[:20]

yolo8 = YOLO(str(YOLOV8_WEIGHTS)) if YOLOV8_WEIGHTS.exists() else None
yolon = YOLO(str(YOLOV11N_WEIGHTS)) if YOLOV11N_WEIGHTS.exists() else None
sahi_m = AutoDetectionModel.from_pretrained(model_type="ultralytics",model_path=str(YOLOV11_WEIGHTS),confidence_threshold=0.25,device=DEVICE if DEVICE!="0" else "cuda:0") if YOLOV11_WEIGHTS.exists() else None

colors = {"gt":(0,255,0),"v8":(255,0,255),"sahi":(0,255,255),"n":(255,255,0)}
def draw_box(img,x1,y1,x2,y2,color,t=5):
    cv2.rectangle(img,(x1,y1),(x2,y2),(0,0,0),t+2)
    cv2.rectangle(img,(x1,y1),(x2,y2),color,t)

compare_dir = RESULTS_DIR/"visual_comparison"
compare_dir.mkdir(parents=True, exist_ok=True)

for idx, ip in enumerate(val_images):
    img_rgb = cv2.cvtColor(cv2.imread(str(ip)),cv2.COLOR_BGR2RGB)
    h,w = img_rgb.shape[:2]
    name = ip.stem
    panels = []
    
    # GT
    gt_img=img_rgb.copy(); gc_=0
    lp=val_lbl_dir/(ip.stem+".txt")
    if lp.exists():
        for ln in lp.read_text().strip().split('\n'):
            pts=ln.split()
            if len(pts)!=5: continue
            _,cx,cy,bw,bh=map(float,pts)
            draw_box(gt_img,int((cx-bw/2)*w),int((cy-bh/2)*h),int((cx+bw/2)*w),int((cy+bh/2)*h),colors["gt"])
            gc_+=1
    panels.append(("Ground Truth",gt_img,gc_))
    
    # v8
    if yolo8:
        vi=img_rgb.copy(); vc=0
        for xy in sv.Detections.from_ultralytics(yolo8.predict(str(ip),conf=0.25,iou=0.4,verbose=False,max_det=900)[0]).xyxy:
            draw_box(vi,*map(int,xy),colors["v8"]); vc+=1
        panels.append(("YOLOv8m",vi,vc))
    
    # SAHI
    if sahi_m:
        si=img_rgb.copy(); sc=0
        res=get_sliced_prediction(str(ip),sahi_m,slice_height=640,slice_width=640,overlap_height_ratio=0.2,overlap_width_ratio=0.2,perform_standard_pred=True,postprocess_match_threshold=0.5,verbose=0)
        for p in res.object_prediction_list:
            b=p.bbox; draw_box(si,int(b.minx),int(b.miny),int(b.maxx),int(b.maxy),colors["sahi"]); sc+=1
        panels.append(("YOLOv11l+SAHI",si,sc))
    
    # v11n
    if yolon:
        ni=img_rgb.copy(); nc=0
        for xy in sv.Detections.from_ultralytics(yolon.predict(str(ip),conf=0.25,iou=0.4,verbose=False,max_det=900)[0]).xyxy:
            draw_box(ni,*map(int,xy),colors["n"]); nc+=1
        panels.append(("YOLOv11n",ni,nc))
    
    for pname, pimg, pcount in panels:
        fig,ax = plt.subplots(figsize=(12,10))
        ax.imshow(pimg); ax.axis("off")
        ax.set_title(f"{name} — {pname} — {pcount} products",fontsize=16,fontweight="bold")
        plt.tight_layout()
        safe = pname.replace('+','_').replace(' ','_')
        plt.savefig(compare_dir/f"{name}_{safe}_{pcount}.png",dpi=150,bbox_inches="tight")
        plt.show()
    
    print(f"  {name}: GT={panels[0][2]}" + "".join(f"  {p[0]}={p[2]}" for p in panels[1:]))
    gc.collect(); torch.cuda.empty_cache()

if yolo8: del yolo8
if yolon: del yolon
if sahi_m: del sahi_m
gc.collect(); torch.cuda.empty_cache()
print(f"\n✓ {len(list(compare_dir.glob('*.png')))} images saved")

---
## 9. Upload Results to S3

In [ ]:
print("Uploading results to S3...\n")
for f in sorted(RESULTS_DIR.glob("*.png")):
    s3.upload_file(str(f), S3_BUCKET, f"{S3_OUTPUT_PREFIX}/results/{f.name}")
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")
compare_dir = RESULTS_DIR/"visual_comparison"
if compare_dir.exists():
    for f in sorted(compare_dir.glob("*.png")):
        s3.upload_file(str(f), S3_BUCKET, f"{S3_OUTPUT_PREFIX}/results/visual_comparison/{f.name}")
        print(f"  visual_comparison/{f.name}")
print(f"\nDownload: https://console.aws.amazon.com/s3/buckets/{S3_BUCKET}?prefix={S3_OUTPUT_PREFIX}/results/")

---
## 10. Export YOLOv11n for Mobile

In [ ]:
# Export the all-data model for mobile deployment
best_n = YOLOV11N_ALL_WEIGHTS if YOLOV11N_ALL_WEIGHTS.exists() else YOLOV11N_WEIGHTS
if best_n.exists():
    model = YOLO(str(best_n))
    try:
        onnx = model.export(format="onnx")
        print(f"✓ ONNX: {onnx}")
    except Exception as e: print(f"ONNX failed: {e}")
    try:
        tflite = model.export(format="tflite")
        print(f"✓ TFLite: {tflite}")
    except Exception as e: print(f"TFLite failed: {e}")
    print(f"\nPyTorch: {best_n.stat().st_size/1e6:.1f} MB")
    del model; gc.collect()
else:
    print("No trained YOLOv11n weights found.")

---
## 11. Final Summary

In [ ]:
print("═"*75); print("  EXPERIMENT COMPLETE"); print("═"*75)
print(f"\n  Dataset: {total_imgs:,} images, {total_boxes:,} boxes")
print(f"  Device:  {GPU_NAME}\n")
for name, w in [("YOLOv8m (baseline, 25%)",YOLOV8_WEIGHTS),("YOLOv11l+SAHI (25%)",YOLOV11_WEIGHTS),("YOLOv11n (25%)",YOLOV11N_WEIGHTS),("YOLOv11n (ALL data)",YOLOV11N_ALL_WEIGHTS)]:
    if w.exists():
        rn = w.parent.parent.name
        print(f"  ✓ {name:30s} → s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/{rn}/weights/best.pt")
    else: print(f"  ⊘ {name}")
print(f"\n  Results: s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/results/")
print("═"*75)

---
## 12. Future: Zero-Shot Models

**Florence-2** (Microsoft) — detection + captioning, 232M params. Could describe products.

**Grounding DINO** (academic) — text-prompted detection, 172M params. No training needed.

**OWLv2** (Google) — multi-query detection, 130M params. Products + price tags.

These would run inference-only on the same test images for comparison.

---
## 13. Future: DINOv2 Product Identification

**Pipeline:** YOLO detects boxes → crop → DINOv2 encodes → FAISS matches against Open Food Facts → product ID.

**Data:** `Nutritionell Version-1-2.ipynb` collected 132K US products from OFF.

**S3:** `s3://amazon-sagemaker-060119698773-us-east-2-4vgskkrpdupdbd/shared/filtered_data/`

See DINOv2 code cells below — run individually when ready.

In [ ]:
# ═══════════════ DINOv2 — DO NOT RUN unless ready ═══════════════
print("DINOv2 cells ready. Run individually when needed.")

### 13a: Load DINOv2

In [ ]:
!pip install -q transformers faiss-cpu
from transformers import AutoModel, AutoImageProcessor
DINOV2_MODEL = "facebook/dinov2-base"
EMBEDDING_DIM = 768
OFF_BUCKET = "amazon-sagemaker-060119698773-us-east-2-4vgskkrpdupdbd"
OFF_FILTERED_PREFIX = "shared/filtered_data/"
OFF_IMAGES_PREFIX = "product_vision_data/Model_1/front/"
OFF_PUBLIC_BUCKET = "openfoodfacts-images"
INDEX_S3_PREFIX = "models/dinov2_product_index"
DINOV2_DIR = PROJECT_ROOT / "dinov2"
DINOV2_DIR.mkdir(parents=True, exist_ok=True)
dinov2_processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL)
dinov2_model = AutoModel.from_pretrained(DINOV2_MODEL)
dinov2_device = "cuda" if torch.cuda.is_available() else "cpu"
dinov2_model = dinov2_model.to(dinov2_device).eval()
print(f"DINOv2 on {dinov2_device}, {sum(p.numel() for p in dinov2_model.parameters())/1e6:.1f}M params")

### 13b: Load OFF Metadata

In [ ]:
try:
    import awswrangler as wr
except ImportError:
    !pip install -q awswrangler
    import awswrangler as wr
off_df = wr.s3.read_parquet(path=f"s3://{OFF_BUCKET}/{OFF_FILTERED_PREFIX}")
print(f"Loaded {len(off_df):,} products")
pcols = [c for c in ["code","derived_product_name","brands","categories","ingredients_text","nova_group","nutriscore_grade","front_imgid"] if c in off_df.columns]
products_with_images = off_df[pcols][off_df["front_imgid"].notna()].reset_index(drop=True) if "front_imgid" in off_df.columns else off_df[pcols].reset_index(drop=True)
print(f"With images: {len(products_with_images):,}")

### 13c: Encoding Helpers

In [ ]:
from botocore.config import Config as BotoConfig
from botocore import UNSIGNED
s3_priv = boto3.client("s3")
s3_pub = boto3.client("s3", config=BotoConfig(signature_version=UNSIGNED))
def get_off_key(code,imgid):
    c=str(code).split(".")[0].zfill(13); i=str(imgid).split(".")[0]
    return f"data/{c[0:3]}/{c[3:6]}/{c[6:9]}/{c[9:]}/{i}.jpg"
def load_s3_img(bucket,key,client):
    return Image.open(io.BytesIO(client.get_object(Bucket=bucket,Key=key)["Body"].read())).convert("RGB")
@torch.no_grad()
def encode_img(pil):
    inp=dinov2_processor(images=pil,return_tensors="pt").to(dinov2_device)
    cls=dinov2_model(**inp).last_hidden_state[:,0,:]
    return torch.nn.functional.normalize(cls,p=2,dim=1).cpu().numpy().flatten()
tr=products_with_images.iloc[0]; tb=str(tr["code"]).split(".")[0].zfill(13)
try:
    ti=load_s3_img(OFF_BUCKET,f"{OFF_IMAGES_PREFIX}{tb}.jpg",s3_priv); IMAGE_SOURCE="private"
except:
    try: ti=load_s3_img(OFF_PUBLIC_BUCKET,get_off_key(tr["code"],tr["front_imgid"]),s3_pub); IMAGE_SOURCE="public"
    except: IMAGE_SOURCE=None
if IMAGE_SOURCE: print(f"✓ {IMAGE_SOURCE}, shape={encode_img(ti).shape}")

### 13d: Encode All Products

In [ ]:
CKPT=500
emb_path=DINOV2_DIR/"embeddings.npy"; meta_path=DINOV2_DIR/"metadata.pkl"
if emb_path.exists() and meta_path.exists():
    ex=np.load(str(emb_path)); 
    with open(meta_path,"rb") as f: em=pickle.load(f)
    si=len(em); all_e=list(ex); all_m=em; print(f"Resume: {si:,}")
else: si=0; all_e=[]; all_m=[]
tot=len(products_with_images); fail=0
for idx in tqdm(range(si,tot),desc="Encoding",initial=si,total=tot):
    row=products_with_images.iloc[idx]; bc=str(row["code"]).split(".")[0].zfill(13)
    try:
        img=load_s3_img(OFF_BUCKET,f"{OFF_IMAGES_PREFIX}{bc}.jpg",s3_priv) if IMAGE_SOURCE=="private" else load_s3_img(OFF_PUBLIC_BUCKET,get_off_key(row["code"],row.get("front_imgid","1")),s3_pub)
        all_e.append(encode_img(img))
        all_m.append({"barcode":bc,"name":row.get("derived_product_name",""),"brands":row.get("brands",""),"categories":row.get("categories",""),"nova_group":row.get("nova_group"),"nutriscore":row.get("nutriscore_grade","")})
    except: fail+=1; continue
    if len(all_m)%CKPT==0 and len(all_m)>si:
        np.save(str(emb_path),np.array(all_e,dtype=np.float32))
        with open(meta_path,"wb") as f: pickle.dump(all_m,f)
ea=np.array(all_e,dtype=np.float32); np.save(str(emb_path),ea)
with open(meta_path,"wb") as f: pickle.dump(all_m,f)
print(f"\n✓ {len(all_m):,} encoded ({fail:,} failed), shape={ea.shape}")

### 13e: Build FAISS Index

In [ ]:
import faiss
if "ea" not in dir(): ea=np.load(str(DINOV2_DIR/"embeddings.npy")); 
if "all_m" not in dir():
    with open(DINOV2_DIR/"metadata.pkl","rb") as f: all_m=pickle.load(f)
fi=faiss.IndexFlatIP(EMBEDDING_DIM); fi.add(ea)
fp=DINOV2_DIR/"product_index.faiss"; faiss.write_index(fi,str(fp))
print(f"✓ FAISS: {fi.ntotal:,} vectors, {fp.stat().st_size/1e6:.1f} MB")

### 13f: Identify Products

In [ ]:
def identify_product(crop,top_k=5,min_conf=0.5):
    q=encode_img(crop).reshape(1,-1); sims,idxs=fi.search(q,top_k)
    return [{**all_m[i],"similarity":round(float(s),4)} for s,i in zip(sims[0],idxs[0]) if 0<=i<len(all_m) and s>=min_conf]

def analyze_shelf(img_path,yolo_wt,conf=0.25,top_k=3,min_sim=0.5):
    ym=YOLO(str(yolo_wt)); res=ym.predict(str(img_path),conf=conf,iou=0.4,verbose=False)
    dets=sv.Detections.from_ultralytics(res[0]); img=cv2.imread(str(img_path)); h,w=img.shape[:2]; out=[]
    for i,(xy,c) in enumerate(zip(dets.xyxy,dets.confidence)):
        x1,y1,x2,y2=map(int,xy); px,py=int((x2-x1)*0.05),int((y2-y1)*0.05)
        crop=img[max(0,y1-py):min(h,y2+py),max(0,x1-px):min(w,x2+px)]
        if crop.size==0: continue
        matches=identify_product(Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB)),top_k=top_k,min_conf=min_sim)
        out.append({"idx":i,"bbox":[x1,y1,x2,y2],"conf":round(float(c),4),"match":matches[0] if matches else None,"identified":len(matches)>0})
    del ym; return out

best_w=next((w for w in [YOLOV11N_ALL_WEIGHTS,YOLOV11_WEIGHTS,YOLOV8_WEIGHTS] if w.exists()),None)
test_sh=sorted((DATA_DIR/"images"/"test").glob("*.jpg"),key=lambda p:int(p.stem.split("_")[1]))[:1]
if best_w and test_sh and fi.ntotal>0:
    r=analyze_shelf(test_sh[0],best_w)
    print(f"Detected {len(r)}, identified {sum(1 for p in r if p['identified'])}")
    for p in r[:5]:
        m=p["match"]
        if m: print(f"  Box {p['idx']}  sim={m['similarity']:.3f}  {m.get('name','?')[:40]}")
else: print("Need YOLO weights + FAISS index.")

### 13g: Save DINOv2 Index to S3

In [ ]:
for fn,desc in [("product_index.faiss","FAISS"),("embeddings.npy","Embeddings"),("metadata.pkl","Metadata")]:
    lp=DINOV2_DIR/fn
    if not lp.exists(): continue
    s3.upload_file(str(lp),S3_BUCKET,f"{INDEX_S3_PREFIX}/{fn}")
    print(f"  ✓ {desc} ({lp.stat().st_size/1e6:.1f} MB) → s3://{S3_BUCKET}/{INDEX_S3_PREFIX}/{fn}")